# FixCheck - Wang Dataset

Runs FixCheck (codellama, 20 prefixes) on all Wang patches and compares against PReMS.

**Config matches the D4C / SRepair / RepairLLaMA / PRISM runs:** six-type input fallback, fast Drive restore, CSV written after every patch.

**Before running:** Runtime -> Change runtime type -> GPU (L4 or A100).

## 1. Upload inputs

Run this first so the files are ready while setup installs.

In [ ]:
from google.colab import files
print('Upload wang_prms_results.csv and wang_patches.zip')
uploaded = files.upload()

Upload wang_prms_results.csv and wang_patches.zip


Saving wang_patches.zip to wang_patches.zip
Saving wang_prms_results.csv to wang_prms_results.csv


## 2. Setup

In [ ]:
%%bash
apt-get update -qq
apt-get install -y -qq openjdk-8-jdk openjdk-11-jdk openjdk-17-jdk \
    git subversion perl cpanminus ant locales wget curl unzip lsof zstd pciutils lshw
locale-gen en_US.UTF-8
echo 'Done'

(Reading database ... 127332 files and directories currently installed.)
Preparing to unpack .../libperl5.38t64_5.38.2-3.2ubuntu0.6_amd64.deb ...
Unpacking libperl5.38t64:amd64 (5.38.2-3.2ubuntu0.6) over (5.38.2-3.2ubuntu0.2) ...
Preparing to unpack .../perl_5.38.2-3.2ubuntu0.6_amd64.deb ...
Unpacking perl (5.38.2-3.2ubuntu0.6) over (5.38.2-3.2ubuntu0.2) ...
Preparing to unpack .../perl-base_5.38.2-3.2ubuntu0.6_amd64.deb ...
Unpacking perl-base (5.38.2-3.2ubuntu0.6) over (5.38.2-3.2ubuntu0.2) ...
Setting up perl-base (5.38.2-3.2ubuntu0.6) ...
(Reading database ... 127332 files and directories currently installed.)
Preparing to unpack .../000-perl-modules-5.38_5.38.2-3.2ubuntu0.6_all.deb ...
Unpacking perl-modules-5.38 (5.38.2-3.2ubuntu0.6) over (5.38.2-3.2ubuntu0.2) ...
Selecting previously unselected package pci.ids.
Preparing to unpack .../001-pci.ids_0.0~2024.03.31-1ubuntu0.1_all.deb ...
Unpacking pci.ids (0.0~2024.03.31-1ubuntu0.1) ...
Selecting previously unselected package libpci

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)


In [ ]:
%%bash
curl -fsSL https://ollama.com/install.sh | sh
echo 'Ollama installed'

Ollama installed


>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> NVIDIA GPU installed.
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [ ]:
import subprocess, time, requests, os

subprocess.run(['pkill', '-f', 'ollama'], capture_output=True)
time.sleep(2)
ollama_env = {
    **os.environ,
    'LD_LIBRARY_PATH': '/usr/lib64-nvidia:/usr/local/cuda-12.8/targets/x86_64-linux/lib:' + os.environ.get('LD_LIBRARY_PATH', ''),
    'CUDA_VISIBLE_DEVICES': '0',
}
subprocess.Popen(['ollama', 'serve'],
    stdout=open('/tmp/ollama.log', 'w'),
    stderr=subprocess.STDOUT, env=ollama_env)
time.sleep(6)
try:
    r = requests.get('http://localhost:11434/api/tags')
    print('Ollama running:', r.status_code)
except Exception as e:
    print('ERROR:', e)
with open('/tmp/ollama.log') as f:
    log = f.read()
    if any(x in log for x in ['Tesla','CUDA','A100','L4']):
        print('GPU detected!')
    print(log[-300:])

Ollama running: 200
GPU detected!
i_id=0000:00:03.0 type=discrete total="22.0 GiB" available="21.8 GiB"
time=2026-10-15T16:53:16.978Z level=INFO source=routes.go:2551 msg="vram-based default context" total_vram="22.0 GiB" default_num_ctx=4096
[GIN] 2026/10/15 - 16:53:20 | 200 |     392.253µs |       127.0.0.1 | GET      "/api/tags"



In [ ]:
%%bash
ollama pull codellama
ollama list

NAME                ID              SIZE      MODIFIED               
codellama:latest    8fdf8f752f6e    3.8 GB    Less than a second ago    


pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest ⠏ pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest ⠏ pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest 
pulling 3a43f93b78ec:   0% ▕                  ▏  17 MB/3.8 GB                  pulling manifest 
pulling 3a43f93b78ec:   3% ▕                  ▏ 110 MB/3.8 GB                  pulling manifest 
pulling 3a43f93b78ec:   4% ▕                  ▏ 161 MB/3.8 GB                  pulling manifest 
pulling 3a43f93b78ec:   7% ▕█                 ▏ 262 MB/3.8 GB                  pulling manifest 
pulling 3a43f93b78ec:   9% ▕█                 ▏ 347 MB/3.8 GB                  pulling manifest 
pulling 3a43f93b78ec:  10% ▕█            

In [ ]:
%%bash
update-alternatives --set java /usr/lib/jvm/java-17-openjdk-amd64/bin/java
update-alternatives --set javac /usr/lib/jvm/java-17-openjdk-amd64/bin/javac
cd /content
git clone https://github.com/facumolina/fixcheck
cd fixcheck
./gradlew shadowJar
ls build/libs/

update-alternatives: using /usr/lib/jvm/java-17-openjdk-amd64/bin/java to provide /usr/bin/java (java) in manual mode
update-alternatives: using /usr/lib/jvm/java-17-openjdk-amd64/bin/javac to provide /usr/bin/javac (javac) in manual mode
...........10%............20%............30%............40%............50%............60%...........70%............80%............90%............100%

Welcome to Gradle 8.0.2!

Here are the highlights of this release:
 - Improvements to the Kotlin DSL
 - Fine-grained parallelism from the first build with configuration cache
 - Configurable Gradle user home cache cleanup

For more details see https://docs.gradle.org/8.0.2/release-notes.html

Starting a Gradle Daemon (subsequent builds will be faster)
> Task :compileJava
> Task :processResources NO-SOURCE
> Task :classes
> Task :shadowJar

BUILD SUCCESSFUL in 21s
2 actionable tasks: 2 executed


Cloning into 'fixcheck'...


In [ ]:
%%bash
pip install -q tokenizers==0.19.1
pip install -q pandas Flask==3.0.3 transformers==4.42.3 scikit-learn
echo 'Done'

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 321.0/321.0 kB 25.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 51.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 484.5 kB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.8/15.8 MB 51.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.7/101.7 kB 11.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.3/9.3 MB 55.7 MB/s eta 0:00:00
Done


  error: subprocess-exited-with-error
  
  × Building wheel for tokenizers (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  ERROR: Failed building wheel for tokenizers
ERROR: ERROR: Failed to build installable wheels for some pyproject.toml based projects (tokenizers)
  error: subprocess-exited-with-error
  
  × Building wheel for tokenizers (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  ERROR: Failed building wheel for tokenizers
ERROR: ERROR: Failed to build installable wheels for some pyproject.toml based projects (tokenizers)


In [ ]:
%%bash
cd /content
git clone https://github.com/rjust/defects4j
update-alternatives --set java /usr/lib/jvm/java-8-openjdk-amd64/jre/bin/java
update-alternatives --set javac /usr/lib/jvm/java-8-openjdk-amd64/bin/javac
cpanm --notest DBI String::Interpolate
cd /content/defects4j
perl ./init.sh
echo 'Defects4J init done'

update-alternatives: using /usr/lib/jvm/java-8-openjdk-amd64/jre/bin/java to provide /usr/bin/java (java) in manual mode
update-alternatives: using /usr/lib/jvm/java-8-openjdk-amd64/bin/javac to provide /usr/bin/javac (javac) in manual mode
--> Working on DBI
Fetching http://www.cpan.org/authors/id/H/HM/HMBRAND/DBI-1.655.tgz ... OK
Configuring DBI-1.655 ... OK
Building DBI-1.655 ... OK
--> Working on String::Interpolate
Fetching http://www.cpan.org/authors/id/N/NE/NEILB/String-Interpolate-0.33.tar.gz ... OK
Configuring String-Interpolate-0.33 ... OK
==> Found dependencies: PadWalker, Safe::Hole
--> Working on PadWalker
Fetching http://www.cpan.org/authors/id/R/RO/ROBIN/PadWalker-2.5.tar.gz ... OK
Configuring PadWalker-2.5 ... OK
Building PadWalker-2.5 ... OK
--> Working on Safe::Hole
Fetching http://www.cpan.org/authors/id/T/TO/TODDR/Safe-Hole-0.17.tar.gz ... OK
Configuring Safe-Hole-0.17 ... OK
Building Safe-Hole-0.17 ... OK
Building String-Interpolate-0.33 ... OK
4 distributions inst

Cloning into 'defects4j'...
Updating files: 100% (10125/10125), done.
curl: (3) URL using bad/illegal format or missing URL
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 7119k  100 7119k    0     0  3752k      0  0:00:01  0:00:01 --:--:-- 3753k
curl: (3) URL using bad/illegal format or missing URL
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
100 17.0M  100 17.0M    0     0  37.5M      0 --:--:-- --:--:-- --:--:-- 37.5M
curl: (3) URL using bad/illegal format or missing URL
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --

In [ ]:
%%bash
sed -i '235s/.*/        # die ("Java 11 is required!\\n\\n"); # patched/' /content/defects4j/framework/core/Constants.pm
grep -n 'Java 11' /content/defects4j/framework/core/Constants.pm

235:        # die ("Java 11 is required!\n\n"); # patched


In [ ]:
import os, subprocess
os.environ['PATH'] = '/content/defects4j/framework/bin:/usr/lib/jvm/java-8-openjdk-amd64/jre/bin:' + os.environ['PATH']
os.environ['JAVA_HOME'] = '/usr/lib/jvm/java-8-openjdk-amd64'
r = subprocess.run(['defects4j', 'info', '-p', 'Math'], capture_output=True, text=True)
print(r.stdout[:200])

Summary of configuration for Project: Math
--------------------------------------------------------------------------------
    Script dir: /content/defects4j/framework
      Base dir: /content/defect


In [ ]:
import zipfile, subprocess
with zipfile.ZipFile('wang_patches.zip', 'r') as z:
    z.extractall('/content/wang_patches')
r = subprocess.run('find /content/wang_patches -name "*.patch" | wc -l',
                   shell=True, capture_output=True, text=True)
print(f'Patch files found: {r.stdout.strip()}')

Patch files found: 1816


## 3. Main loop

Resumes from Drive automatically - completed patches are skipped. Safe to re-run after a crash or disconnect.

In [ ]:
import os, re, shutil, subprocess, pandas as pd, time, requests
from sklearn.metrics import roc_auc_score

os.chdir('/content')

subprocess.run(['pkill', '-f', 'ollama'], capture_output=True)
time.sleep(2)
ollama_env = {
    **os.environ,
    'LD_LIBRARY_PATH': '/usr/lib64-nvidia:/usr/local/cuda-12.8/targets/x86_64-linux/lib:' + os.environ.get('LD_LIBRARY_PATH', ''),
    'CUDA_VISIBLE_DEVICES': '0',
}
subprocess.Popen(['ollama', 'serve'],
    stdout=open('/tmp/ollama.log', 'w'),
    stderr=subprocess.STDOUT, env=ollama_env, start_new_session=True)
time.sleep(6)
try:
    r = requests.get('http://localhost:11434/api/tags')
    print('Ollama running:', r.status_code)
except Exception as e:
    print('Ollama ERROR:', e)

subprocess.run('update-alternatives --set java /usr/lib/jvm/java-8-openjdk-amd64/jre/bin/java', shell=True)
subprocess.run('update-alternatives --set javac /usr/lib/jvm/java-8-openjdk-amd64/bin/javac', shell=True)
subprocess.run("sed -i '235s/.*/        # die (\"Java 11 is required!\\\\n\\\\n\"); # patched/' /content/defects4j/framework/core/Constants.pm", shell=True)

JAVA8     = '/usr/lib/jvm/java-8-openjdk-amd64/bin'
JAVA8_JRE = '/usr/lib/jvm/java-8-openjdk-amd64/jre/bin'
JAVA17    = '/usr/lib/jvm/java-17-openjdk-amd64/bin'
D4J       = '/content/defects4j/framework/bin'
BASE      = '/usr/local/sbin:/usr/local/bin:/usr/sbin:/usr/bin:/sbin:/bin'

setup_env = {**os.environ,
             'PATH': f'{JAVA8_JRE}:{JAVA8}:{D4J}:{BASE}',
             'JAVA_HOME': '/usr/lib/jvm/java-8-openjdk-amd64'}
run_env   = {**os.environ,
             'PATH': f'{JAVA17}:{D4J}:{BASE}',
             'JAVA_HOME': '/usr/lib/jvm/java-17-openjdk-amd64',
             'FIXCHECK': '/content/fixcheck',
             'DEFECT_REPAIRING_DATASET': '/content/DefectRepairing'}

r = subprocess.run('defects4j info -p Math', shell=True, capture_output=True, text=True, env=setup_env)
print('D4J check:', r.stdout[:50] if r.returncode==0 else r.stderr[:200])

prems_df = pd.read_csv('wang_prms_results.csv')
prems_df = prems_df.dropna(subset=['bug_id']).copy()
prems_df['patch_file'] = prems_df['patch_file'].str.replace(
    '/workspaces/wang-prms-experiment/wang_patches',
    '/content/wang_patches/wang_patches', regex=False)
prems_df['project'] = prems_df['bug_id'].str.split('-').str[0]
prems_df['bug']     = prems_df['bug_id'].str.split('-').str[1]
prems_df['patch_num'] = prems_df['patch_file'].apply(
    lambda x: re.search(r'patch(\d+)-', str(x), re.IGNORECASE).group(1) if pd.notna(x) else None)
prems_df['patch_uid'] = (prems_df['project'] + '_' + prems_df['bug'] + '_' +
                         prems_df['tool'] + '_' + prems_df['patch_num'].fillna(''))

run_df = prems_df[prems_df['status'] == 'OK'].copy().reset_index(drop=True)
print(f'OK patches: {len(run_df)}')

_k = pd.to_numeric(run_df['killed'],   errors='coerce').fillna(0)
_s = pd.to_numeric(run_df['survived'], errors='coerce').fillna(0)
run_df = run_df[(_k + _s) > 0].reset_index(drop=True)

print(f'Patches to run: {len(run_df)}')
print(f'Label dist: {run_df["label"].value_counts().to_dict()}')
print(f'Missing patch files: {run_df["patch_file"].apply(lambda x: not os.path.exists(str(x))).sum()}')

from google.colab import drive
drive.mount('/content/drive')
DRIVE_BACKUP = '/content/drive/MyDrive/fixcheck-wang'
LOCAL_OUTPUT = '/content/fixcheck/fixcheck-output/defects-repairing'
TMP_BASE     = '/content/DefectRepairing/tmp/base'
CSV_PATH     = f'{DRIVE_BACKUP}/wang_prems_vs_fixcheck.csv'

prev_status = {}
if os.path.exists(CSV_PATH):
    _p = pd.read_csv(CSV_PATH)
    prev_status = dict(zip(_p['patch_uid'], _p['fixcheck_status']))
attempted = {u for u, s in prev_status.items() if s != 'NOT_RUN'}
print(f'Already attempted (will skip): {len(attempted)}')

os.makedirs(DRIVE_BACKUP, exist_ok=True)
os.makedirs(LOCAL_OUTPUT, exist_ok=True)
os.makedirs(TMP_BASE, exist_ok=True)

restored = 0
for folder in os.listdir(DRIVE_BACKUP):
    src_report = f'{DRIVE_BACKUP}/{folder}/codellama/report.csv'
    if os.path.exists(src_report):
        dst_dir = f'{LOCAL_OUTPUT}/{folder}/codellama'
        os.makedirs(dst_dir, exist_ok=True)
        shutil.copy(src_report, f'{dst_dir}/report.csv')
        restored += 1
print(f'Restored {restored} report.csv files from Drive')

done = [d for d in os.listdir(LOCAL_OUTPUT)
        if os.path.exists(f'{LOCAL_OUTPUT}/{d}/codellama/report.csv')]
print(f'Already done: {len(done)}')

for d in os.listdir(TMP_BASE):
    if not os.path.exists(os.path.join(TMP_BASE, d, '.trigger_test')):
        shutil.rmtree(os.path.join(TMP_BASE, d), ignore_errors=True)

def save_csv(results):
    rows = []
    results_dict = {r['patch_uid']: r for r in results}
    for _, row in run_df.iterrows():
        patch_uid   = row['patch_uid']
        report_path = f'{LOCAL_OUTPUT}/{patch_uid}/codellama/report.csv'
        res = results_dict.get(patch_uid)
        status = res['status'] if res else ('DONE' if os.path.exists(report_path) else 'NOT_RUN')
        if not os.path.exists(report_path):
            rows.append({'patch_uid': patch_uid, 'bug_id': row['bug_id'],
                        'tool': row['tool'], 'label': int(row['label']),
                        'prmsc': row['prmsc'], 'fixcheck_status': status,
                        'fixcheck_failing': None, 'fixcheck_pred': None})
            continue
        try:
            report = pd.read_csv(report_path)
            failing = int(report['assertion_failing_prefixes'].sum())
            fixcheck_pred = 0 if failing > 0 else 1
        except:
            failing, fixcheck_pred = None, None
        rows.append({'patch_uid': patch_uid, 'bug_id': row['bug_id'],
                    'tool': row['tool'], 'label': int(row['label']),
                    'prmsc': row['prmsc'], 'fixcheck_status': 'DONE',
                    'fixcheck_failing': failing, 'fixcheck_pred': fixcheck_pred})
    df = pd.DataFrame(rows)
    df.to_csv(CSV_PATH, index=False)
    a = df[df['fixcheck_pred'].notna()].copy()
    a['label'] = pd.to_numeric(a['label'], errors='coerce')
    a['prmsc'] = pd.to_numeric(a['prmsc'], errors='coerce')
    a['fixcheck_failing'] = pd.to_numeric(a['fixcheck_failing'], errors='coerce')
    a = a.dropna(subset=['label','prmsc','fixcheck_failing'])
    if len(a) > 1 and a['label'].nunique() > 1:
        try:
            auc_fc = roc_auc_score(1 - a['label'], a['fixcheck_failing'])
            auc_pr = roc_auc_score(1 - a['label'], 1 - a['prmsc'])
            print(f'  [CSV saved - DONE:{len(a)} | FixCheck AUC:{auc_fc:.3f} | PReMS AUC:{auc_pr:.3f}]')
        except:
            print(f'  [CSV saved - DONE:{len(a)}]')
    return df

def ensure_ollama():
    try:
        requests.get('http://localhost:11434/api/tags', timeout=3)
        return True
    except:
        print('  [Ollama down, restarting...]', end=' ', flush=True)
        subprocess.run(['pkill', '-f', 'ollama'], capture_output=True)
        time.sleep(2)
        subprocess.Popen(['ollama', 'serve'], stdout=open('/tmp/ollama.log', 'w'),
                        stderr=subprocess.STDOUT, env=ollama_env, start_new_session=True)
        time.sleep(6)
        try:
            requests.get('http://localhost:11434/api/tags', timeout=3)
            print('[Ollama back up]', end=' ', flush=True)
            return True
        except:
            print('[Ollama FAILED]')
            return False

def checkout_bug(project, bug):
    bug_base     = os.path.join(TMP_BASE, f'{project}{bug}b')
    trigger_file = os.path.join(bug_base, '.trigger_test')
    if os.path.exists(trigger_file):
        return open(trigger_file).read().strip(), None
    if not os.path.isdir(bug_base):
        cmd = f'defects4j checkout -p {project} -v {bug}b -w {bug_base}'
        r = subprocess.run(cmd, shell=True, capture_output=True, text=True,
                          timeout=300, env=setup_env)
        if r.returncode != 0:
            return None, f'CHECKOUT_FAILED: {r.stderr[-100:]}'
        os.chdir(bug_base)
        subprocess.run('defects4j compile', shell=True, capture_output=True,
                      env=setup_env, timeout=180)
    else:
        os.chdir(bug_base)
    r2 = subprocess.run('defects4j export -p tests.trigger',
                       shell=True, capture_output=True, text=True,
                       env=setup_env, timeout=60)
    trigger = r2.stdout.strip().split('\n')[0] if r2.stdout.strip() else None
    if not trigger:
        return None, 'NO_TRIGGER'
    with open(trigger_file, 'w') as f:
        f.write(trigger)
    return trigger, None

def parse_trigger(trigger):
    if '::' in trigger: cls, method = trigger.split('::', 1)
    elif '#' in trigger: cls, method = trigger.split('#', 1)
    else: cls, method = trigger, ''
    return cls.strip(), method.strip()

def detect_input_class(bug_base, trigger_class, trigger_method):
    class_file = trigger_class.split('.')[-1] + '.java'
    for root, dirs, files in os.walk(bug_base):
        if class_file in files:
            try:
                content = open(os.path.join(root, class_file)).read()
                idx = content.find(f'void {trigger_method}')
                if idx >= 0:
                    brace = content.find('{', idx)
                    if brace >= 0:
                        depth, body = 0, ''
                        for ch in content[brace:]:
                            if ch == '{': depth += 1
                            elif ch == '}': depth -= 1
                            body += ch
                            if depth == 0: break
                        content = body
                if re.search(r'\b\d+\.\d+\b', content) or 'double' in content: return 'double'
                if re.search(r'\b\d+L\b', content) or 'long' in content: return 'long'
                if '"' in content or 'String' in content: return 'java.lang.String'
                if re.search(r'\b\d+\b', content) or 'int' in content: return 'int'
                if 'true' in content or 'false' in content or 'boolean' in content: return 'boolean'
            except: pass
    return 'java.lang.Object'

INPUT_TYPES = ['int', 'double', 'long', 'java.lang.String', 'boolean', 'java.lang.Object']

def get_classpaths(working_dir):
    if os.path.isdir(os.path.join(working_dir, 'target/classes')):
        if os.path.isdir(os.path.join(working_dir, 'src/test/java')): tests_src = 'src/test/java'
        elif os.path.isdir(os.path.join(working_dir, 'src/test')): tests_src = 'src/test'
        else: tests_src = 'src/test/java'
        return 'target/classes:target/test-classes', 'target/test-classes', tests_src
    if os.path.isdir(os.path.join(working_dir, 'build')):
        return 'build:build-tests', 'build-tests', 'tests'
    return 'target/classes:target/test-classes', 'target/test-classes', 'src/test/java'

def run_fixcheck(patch_uid, working_dir, trigger_class, trigger_method,
                 main_dep, tests_build, tests_src, input_class):
    os.chdir('/content/fixcheck')
    subject_cp        = ':'.join([os.path.join(working_dir, d) for d in main_dep.split(':')])
    test_classes_path = os.path.join(working_dir, tests_build)
    target_test_dir   = os.path.join(working_dir, tests_src)
    failure_log       = os.path.join(working_dir, 'failing_tests')
    with open(failure_log, 'w') as f:
        f.write(f'--- {trigger_class}\n')
        f.write(f'{trigger_class}::{trigger_method}\n')
    props_file = f'/tmp/fixcheck-{patch_uid[:50]}.properties'
    with open(props_file, 'w') as f:
        f.write(f'test-classes-path={test_classes_path}\n')
        f.write(f'test-class={trigger_class}\n')
        f.write(f'test-methods={trigger_method}\n')
        f.write(f'test-classes-src={target_test_dir}\n')
        f.write(f'test-failure-trace-log={failure_log}\n')
        f.write(f'inputs-class={input_class}\n')
        f.write(f'number-of-prefixes=20\n')
        f.write(f'assertion-generator=codellama\n')
    full_cp = f'build/libs/fixcheck-all-1.0.0.jar:{subject_cp}:{test_classes_path}'
    cmd = (f'/usr/lib/jvm/java-17-openjdk-amd64/bin/java '
           f'-cp {full_cp} org.imdea.fixcheck.FixCheck -p {props_file}')
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True,
                      timeout=600, env=run_env)
    output_dir = f'fixcheck-output/defects-repairing/{patch_uid}/codellama'
    os.makedirs(output_dir, exist_ok=True)
    for fname in ['report.csv', 'scores-failing-tests.csv', 'log.out']:
        src = f'fixcheck-output/{fname}'
        if os.path.exists(src): subprocess.run(f'mv {src} {output_dir}/', shell=True)
    for dname in ['failing-tests', 'passing-tests', 'non-compiling-tests']:
        src = f'fixcheck-output/{dname}'
        if os.path.exists(src): subprocess.run(f'mv {src} {output_dir}/', shell=True)
    return r.stdout + r.stderr

def backup_report(patch_uid):
    report_src = f'{LOCAL_OUTPUT}/{patch_uid}/codellama/report.csv'
    if os.path.exists(report_src):
        dst_dir = f'{DRIVE_BACKUP}/{patch_uid}/codellama'
        os.makedirs(dst_dir, exist_ok=True)
        shutil.copy(report_src, f'{dst_dir}/report.csv')

save_csv([])

results = []

for i, row in run_df.iterrows():
    os.chdir('/content')
    patch_uid  = row['patch_uid']
    project    = row['project']
    bug        = row['bug']
    patch_file = row['patch_file']

    report_file = f'{LOCAL_OUTPUT}/{patch_uid}/codellama/report.csv'
    if os.path.exists(report_file):
        print(f'[{i+1}/{len(run_df)}] SKIP {patch_uid}')
        results.append({'patch_uid': patch_uid, 'bug_id': row['bug_id'],
                       'tool': row['tool'], 'label': row['label'],
                       'prmsc': row['prmsc'], 'status': 'SKIPPED'})
        continue

    if patch_uid in attempted:
        print(f'[{i+1}/{len(run_df)}] SKIP {patch_uid} ({prev_status[patch_uid]})')
        results.append({'patch_uid': patch_uid, 'bug_id': row['bug_id'],
                       'tool': row['tool'], 'label': row['label'],
                       'prmsc': row['prmsc'], 'status': prev_status[patch_uid]})
        continue

    print(f'[{i+1}/{len(run_df)}] {patch_uid}', end=' ', flush=True)
    ensure_ollama()

    try:
        trigger, err = checkout_bug(project, bug)
    except subprocess.TimeoutExpired:
        print('CHECKOUT_TIMEOUT')
        results.append({'patch_uid': patch_uid, 'bug_id': row['bug_id'],
                       'tool': row['tool'], 'label': row['label'],
                       'prmsc': row['prmsc'], 'status': 'CHECKOUT_TIMEOUT'})
        save_csv(results)
        continue
    if err:
        print(f'ERR: {err[:80]}')
        results.append({'patch_uid': patch_uid, 'bug_id': row['bug_id'],
                       'tool': row['tool'], 'label': row['label'],
                       'prmsc': row['prmsc'], 'status': err[:50]})
        save_csv(results)
        continue

    trigger_class, trigger_method = parse_trigger(trigger)
    bug_base = os.path.join(TMP_BASE, f'{project}{bug}b')
    input_class = detect_input_class(bug_base, trigger_class, trigger_method)
    print(f'input={input_class}', end=' ', flush=True)

    working_dir = f'/content/DefectRepairing/tmp/{patch_uid}/{project}{bug}b'
    os.makedirs(os.path.dirname(working_dir), exist_ok=True)
    if not os.path.isdir(working_dir):
        shutil.copytree(bug_base, working_dir)

    os.chdir(working_dir)
    r_patch = subprocess.run(f'patch -u -p1 < {patch_file}',
                            shell=True, capture_output=True, text=True)
    if r_patch.returncode != 0:
        print('PATCH_FAILED')
        results.append({'patch_uid': patch_uid, 'bug_id': row['bug_id'],
                       'tool': row['tool'], 'label': row['label'],
                       'prmsc': row['prmsc'], 'status': 'PATCH_FAILED'})
        shutil.rmtree(working_dir, ignore_errors=True)
        save_csv(results)
        continue

    os.chdir(working_dir)
    r_compile = subprocess.run('defects4j compile', shell=True,
                              capture_output=True, text=True,
                              env=setup_env, timeout=180)
    combined_compile = r_compile.stdout + r_compile.stderr
    if 'FAILED' in combined_compile or r_compile.returncode != 0:
        print('COMPILE_FAILED')
        results.append({'patch_uid': patch_uid, 'bug_id': row['bug_id'],
                       'tool': row['tool'], 'label': row['label'],
                       'prmsc': row['prmsc'], 'status': 'COMPILE_FAILED'})
        shutil.rmtree(working_dir, ignore_errors=True)
        save_csv(results)
        continue

    main_dep, tests_build, tests_src = get_classpaths(working_dir)
    print('RUN...', end=' ', flush=True)

    final_status = 'NO_LOCALS'
    for ic in INPUT_TYPES:
        try:
            combined = run_fixcheck(patch_uid, working_dir, trigger_class, trigger_method,
                                   main_dep, tests_build, tests_src, ic)
        except subprocess.TimeoutExpired:
            print('TIMEOUT')
            subprocess.run('pkill -f "fixcheck-all" || true', shell=True)
            final_status = 'TIMEOUT'
            break

        if os.path.exists(report_file):
            print(f'DONE(input={ic})', end=' ', flush=True)
            final_status = 'DONE'
            backup_report(patch_uid)
            print('(backed up)', end=' ', flush=True)
            break
        elif 'No locals of type' in combined:
            continue
        elif 'result" is null' in combined:
            print('TEST_CRASH')
            final_status = 'TEST_CRASH'
            break
        elif 'compile' in combined.lower() and 'failed' in combined.lower():
            print('COMPILE_FAILED')
            final_status = 'COMPILE_FAILED'
            break
        else:
            print('UNKNOWN_FAIL')
            print(f'  tail: {combined[-150:]}')
            final_status = 'UNKNOWN_FAIL'
            break
    else:
        print('NO_LOCALS(all types)')

    results.append({'patch_uid': patch_uid, 'bug_id': row['bug_id'],
                   'tool': row['tool'], 'label': row['label'],
                   'prmsc': row['prmsc'], 'status': final_status})
    shutil.rmtree(working_dir, ignore_errors=True)
    save_csv(results)

results_df = pd.DataFrame(results)
print('\n=== Final Summary ===')
print(results_df['status'].value_counts())
final_df = save_csv(results)

Ollama running: 200
D4J check: Summary of configuration for Project: Math
-------
OK patches: 394
Patches to run: 243
Label dist: {0: 201, 1: 42}
Missing patch files: 0
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Already attempted (will skip): 154
Restored 168 report.csv files from Drive
Already done: 168
  [CSV saved - DONE:134 | FixCheck AUC:0.544 | PReMS AUC:0.723]
[1/243] SKIP Math_58_Arja_1
[2/243] SKIP Math_4_AVATAR_1
[3/243] SKIP Math_59_AVATAR_1 (TEST_CRASH)
[4/243] SKIP Math_70_Cardumen_1 (UNKNOWN_FAIL)
[5/243] SKIP Math_30_FixMiner_1
[6/243] SKIP Math_34_FixMiner_1
[7/243] SKIP Math_57_FixMiner_1
[8/243] SKIP Math_70_FixMiner_1 (UNKNOWN_FAIL)
[9/243] SKIP Math_79_FixMiner_1
[10/243] SKIP Math_70_jGenProg_1 (UNKNOWN_FAIL)
[11/243] SKIP Math_82_jMutRepair_1
[12/243] SKIP Math_85_jMutRepair_1 (PATCH_FAILED)
[13/243] SKIP Math_58_kPAR_1
[14/243] SKIP Math_70_kPAR_1 (UNKNOWN_FAIL)
[15/243] SKIP Ma

## 4. Results

In [ ]:
from sklearn.metrics import roc_auc_score, matthews_corrcoef, confusion_matrix
import pandas as pd

df = pd.read_csv(CSV_PATH)
a = df[df['fixcheck_pred'].notna()].copy()
a['label'] = pd.to_numeric(a['label'], errors='coerce')
a['prmsc'] = pd.to_numeric(a['prmsc'], errors='coerce')
a['fixcheck_failing'] = pd.to_numeric(a['fixcheck_failing'], errors='coerce')
a['fixcheck_pred'] = pd.to_numeric(a['fixcheck_pred'], errors='coerce')
a = a.dropna(subset=['label','prmsc','fixcheck_failing','fixcheck_pred'])

print(f'Total patches:  {len(df)}')
print(f'Assessable:     {len(a)}')
print(f'  overfitting (label=0): {(a["label"]==0).sum()}')
print(f'  correct     (label=1): {(a["label"]==1).sum()}')
print()
print(df['fixcheck_status'].value_counts())
print()

auc_fc = roc_auc_score(1 - a['label'], a['fixcheck_failing'])
auc_pr = roc_auc_score(1 - a['label'], 1 - a['prmsc'])
mcc_fc = matthews_corrcoef(a['label'], a['fixcheck_pred'])
mcc_pr = matthews_corrcoef(a['label'], (a['prmsc'] >= 0.5).astype(int))

print(f'FixCheck  AUC: {auc_fc:.3f}   MCC: {mcc_fc:.3f}')
print(f'PReMS     AUC: {auc_pr:.3f}   MCC: {mcc_pr:.3f}')
print()

cm = confusion_matrix(a['label'], a['fixcheck_pred'])
print('FixCheck confusion matrix (rows=actual, cols=predicted):')
print('            pred=0  pred=1')
print(f'actual=0 {cm[0,0]:8d}{cm[0,1]:8d}   (overfitting)')
print(f'actual=1 {cm[1,0]:8d}{cm[1,1]:8d}   (correct)')

Total patches:  243
Assessable:     175
  overfitting (label=0): 151
  correct     (label=1): 24

fixcheck_status
DONE            175
UNKNOWN_FAIL     33
TIMEOUT          13
TEST_CRASH       11
PATCH_FAILED     11
Name: count, dtype: int64

FixCheck  AUC: 0.616   MCC: 0.127
PReMS     AUC: 0.756   MCC: 0.263

FixCheck confusion matrix (rows=actual, cols=predicted):
            pred=0  pred=1
actual=0      102      49   (overfitting)
actual=1       12      12   (correct)


## 5. Download and free the GPU

In [ ]:
from google.colab import files, runtime
files.download(CSV_PATH)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Disconnecting runtime to free GPU...
